# 2026-03-19 (목) RAG 문서 처리 기초 - Document, DocumentStore, 클래스 설계

## W2 Day 3: 문서 전처리 & 간단한 RAG 파이프라인 만들기

### 오늘 배우는 것

어제까지 배운 **임베딩(Embedding)**은 문서를 숫자 벡터로 바꿔서 유사도를 구하는 기술이었다.
오늘은 이 임베딩을 **실제 RAG 시스템**에 녹여넣는 과정을 본다.

1. RAG가 무엇인지 개념 정리 (Retrieval + Augmented + Generation)
2. 다양한 파일 포맷(txt, csv, json) 로딩
3. `langchain_core.documents.Document` 객체 사용법
4. 파이썬 **클래스** 복습 → `DocumentStore` 클래스 만들기
5. `retrieve` + `generate`를 합친 Mini RAG 완성

### 비유로 이해하는 RAG

> ChatGPT는 **시험장에서 커닝 페이퍼 없이 답을 쓰는 학생**이다.
> 자기가 외운 범위 밖의 질문(예: 우리 회사 재택근무 규정)이 나오면 그럴듯하게 지어낸다 → **환각(Hallucination)**.
>
> RAG는 그 학생에게 **오픈북 시험**을 허용하는 것이다.
> 질문 → 관련 페이지 검색(Retrieval) → 책 내용을 보고(Augmented) → 답 쓰기(Generation).

### RAG vs Fine-tuning

| 항목 | RAG (오픈북) | Fine-tuning (재교육) |
|---|---|---|
| 비용 | 저렴 | 비쌈 (재학습 필요) |
| 구현 | 문서만 잘 쪼개면 됨 | 데이터셋 + GPU 필요 |
| 업데이트 | 문서 갈아끼우기만 하면 됨 | 다시 학습해야 함 |
| 한계 | cosine similarity 한계, 시간/인과관계 약함 | 덜 유연함 |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w2_token_embedding_vector_rag/llm_260319_RAG_Document.ipynb)

## 0. Colab 환경 세팅

Colab에서 실행할 때 필요한 라이브러리를 설치한다.
`langchain-text-splitters`는 문서를 청크 단위로 나누는 도구,
`langchain-community`는 FAISS 같은 커뮤니티 벡터스토어를 포함한다.

In [ ]:
# Colab 환경 전용 설치 스크립트 (로컬 실행 시 주석 처리)
!pip install -q langchain-text-splitters
!pip install -q langchain-openai
!pip install -q langchain_classic
!pip install -q langchain-community
!pip install -q scikit-learn

## 1. 라이브러리 import & API 키 설정

오늘 다룰 파일 종류: `txt`, `csv`, `json` → 관련 라이브러리를 모두 import한다.
- `pathlib.Path`: 경로를 객체처럼 다룰 수 있어 `/` 연산자로 경로 조합이 가능
- `cosine_similarity`: 쿼리 벡터와 문서 벡터의 방향 유사도(= 의미 유사도) 계산
- `HumanMessage`, `SystemMessage`: LangChain에서 역할 기반 프롬프트를 만들 때 사용

In [ ]:
import os
import json
import csv
import textwrap
from pathlib import Path  # 경로 객체처럼 다루기 (Path / 'foo.txt' 가능)
from datetime import datetime

# LangChain 핵심 컴포넌트
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document  # RAG의 표준 문서 객체
from langchain_core.messages import HumanMessage, SystemMessage  # 역할 기반 메시지
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# 코사인 유사도 (벡터 간 각도 기반 유사도)
from sklearn.metrics.pairwise import cosine_similarity

# Colab 전용 - API 키 로드
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

In [ ]:
# LLM(생성 모델)과 Embedding(벡터화 모델)은 다른 모델!
# - gpt-4o-mini: 답변 생성용 (Generation)
# - text-embedding-3-small: 문서를 벡터로 바꿔주는 임베딩 모델 (1536차원)
llm = ChatOpenAI(model='gpt-4o-mini', api_key=api_key)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small', api_key=api_key)

## 2. 샘플 데이터 만들기

실제 사내 문서 대신 연습용 샘플 텍스트를 `sample_data/` 폴더에 만든다.
- 사내 규정(company_policy.txt)
- AI 산업 보고서(ai_report.txt)
- 제품 매뉴얼(product_manual.txt)

`Path('sample_data')` → 이후 `SAMPLE_DIR / 'foo.txt'`처럼 `/` 연산자로 쓸 수 있다.
OS에 상관없이 동작(Windows/Mac/Linux) → `pathlib`을 쓰는 이유.

In [ ]:
# 샘플 디렉토리 생성 (이미 있으면 그대로 두기)
SAMPLE_DIR = Path('sample_data')
SAMPLE_DIR.mkdir(exist_ok=True)

In [ ]:
# 샘플 문서 3종 (딕셔너리로 파일명: 내용 쌍으로 관리)
sample_texts = {
    'company_policy.txt': """주식회사 모두의연구소 사내 규정

제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.

제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.

제3조 (휴가)
연차휴가는 근로기준법에 따라 부여한다.
경조사 휴가는 별도 규정에 따른다.
자기개발 휴가를 연 5일 추가 부여한다.

제4조 (교육)
모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
온라인 학습 플랫폼 이용료를 전액 지원한다.
""",
    'ai_report.txt': """2024년 인공지능 산업 동향 보고서

개요
2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.
특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.

주요 트렌드
RAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.
멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.
AI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.
소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.

시장 전망
2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.
특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.
""",
    'product_manual.txt': """스마트 홈 허브 v3.0 사용자 매뉴얼

제품 소개
스마트 홈 허브 v3.0은 AI 기반 홈 자동화 컨트롤러입니다.
음성 인식, 자동 스케줄링, 에너지 최적화 기능을 제공합니다.

초기 설정
Step 1: 전원을 연결하고 Wi-Fi 네트워크에 접속합니다.
Step 2: 모바일 앱을 설치하고 QR 코드를 스캔합니다.
Step 3: 연동할 IoT 기기를 검색하고 등록합니다.

주요 기능
음성 명령: \"허브야, 거실 조명 켜줘\" 등의 자연어 명령 지원
자동 스케줄: 시간대별 기기 자동 제어
에너지 모니터링: 실시간 전력 사용량 확인 및 절약 팁 제공
보안 모드: 외출 시 자동 보안 설정
"""
}

In [ ]:
# 딕셔너리를 순회하며 실제 파일로 저장 (UTF-8 인코딩 필수 - 한글 깨짐 방지)
for filename, content in sample_texts.items():
    (SAMPLE_DIR / filename).write_text(content, encoding='utf-8')
    # 결과: sample_data/company_policy.txt, sample_data/ai_report.txt, ...

In [ ]:
# CSV 샘플 데이터 (직원 명부)
csv_data = [
    {'이름': '김철수', '부서': '개발1팀', '직급': '대리'},
    {'이름': '김민아', '부서': '개발1팀', '직급': '대리'},
    {'이름': '박지민', '부서': '개발1팀', '직급': '대리'},
]

# DictWriter: 딕셔너리 키를 CSV 컬럼명으로 자동 매핑
with open(SAMPLE_DIR / 'employees.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['이름', '부서', '직급'])
    writer.writeheader()  # 헤더 먼저 작성
    writer.writerows(csv_data)  # 데이터 한번에 작성

## 3. RAG 개념 다시 보기

**RAG = Retrieval + Augmented + Generation**

- **Retrieval (검색)**: 사용자 질문과 가장 관련 있는 문서 찾기
- **Augmented (증강)**: 찾은 문서를 프롬프트에 끼워 넣기
- **Generation (생성)**: 증강된 프롬프트로 LLM이 답변 생성

### 왜 필요한가?

GPT는 **토큰 단위**로 글을 생성한다. "이름이 ___" 다음 올 단어 후보 중 확률이 높은 것을 고른다.
이전 문장의 맥락이 다음 단어에 영향을 주기 때문에, **학습 데이터에 없는 것**(예: 우리 회사 직원 이름)은 **지어낸다(Hallucination)**.

RAG는 이를 방지하기 위해 **내부 검색엔진**을 하나 붙이는 것이라고 생각하면 된다.

### RAG의 한계

- cosine similarity 기반이라 **검색 결과가 기대와 다를 수 있음**
- **시간/인과관계** 같은 복잡한 추론에 약함
- **문서 전처리(Chunking)를 빡세게 해야** 성능이 나옴 → 오늘 수업의 주제

In [ ]:
# 가장 기본적인 RAG 검색 함수 구현하기
# 작은 knowledge_base 리스트를 두고 query와 가장 유사한 문서를 찾는다
knowledge_base = [
    {'id': 1, 'content': '파이썬은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다. 간결한 문법과 풍부한 라이브러리가 특징입니다.', 'source': 'programming_guide.txt'},
    {'id': 2, 'content': 'RAG는 Retrieval-Augmented Generation의 약자로, LLM이 외부 데이터를 참조하여 답변을 생성하는 기술입니다.', 'source': 'ai_glossary.txt'},
    {'id': 3, 'content': '벡터 데이터베이스는 텍스트를 숫자 벡터로 변환하여 저장하고, 유사도 검색을 빠르게 수행하는 데이터베이스입니다.', 'source': 'database_manual.txt'},
]

query = 'RAG 기술이 뭐예요?'

def retrieve_by_similarity(query, documents):
    """쿼리와 문서 리스트를 받아 유사도 순으로 정렬해 반환"""
    # 1단계: 문서 내용만 추출
    contents = [doc['content'] for doc in documents]

    # 2단계: 쿼리와 문서를 모두 벡터화
    query_vec = embeddings.embed_query(query)          # 단일 쿼리용
    doc_vecs = embeddings.embed_documents(contents)    # 여러 문서용 (배치 처리)

    # 3단계: 코사인 유사도 계산 ([0] 이유: 1xN 행렬이 나와서 첫 행만 뽑음)
    scores = cosine_similarity([query_vec], doc_vecs)[0]

    # 4단계: (문서, 점수) 쌍으로 묶어 점수 내림차순 정렬
    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return [(doc, float(score)) for doc, score in ranked]

# 가장 기본적인 RAG 처리 방식
retrieve_by_similarity(query, knowledge_base)

## 4. 파이썬 클래스 복습

RAG 시스템을 함수만으로 짜면 매번 같은 변수(문서 리스트, 임베딩 모델)를 넘겨야 해서 지저분하다.
→ 관련된 변수와 함수를 **하나로 묶은 설계도**가 **클래스(Class)**.

### 비유

> 클래스 = **붕어빵 틀** (설계도)
> 인스턴스 = **틀에서 찍어낸 붕어빵** (실제 객체)
> `self` = **이 붕어빵의 속(팥/슈크림)을 꺼낼 때 쓰는 손잡이**

### 핵심 문법

- `class ClassName:` → 설계도 정의
- `def __init__(self):` → 인스턴스 생성 시 자동 실행 (초기화)
- `self.변수` → 모든 메서드에서 공유되는 인스턴스 변수
- 그냥 `변수` (지역변수) → 함수 안에서만 쓰임

In [ ]:
# 계산기 클래스로 연습 - self가 있으면 인스턴스 전체에서 접근 가능
class Calculator:
    def __init__(self):
        # __init__ = 생성자. 인스턴스 만들 때 자동 실행
        self.history = []  # 히스토리를 빈 리스트로 초기화

    def add(self, a, b):
        # 주의: a, b는 지역변수 (이 메서드 안에서만 살아있음)
        # self.history는 인스턴스 변수 → 다른 메서드에서도 접근 가능
        result = a + b
        self.history.append(f'{a} + {b} = {result}')
        return result

    def get_history(self):
        return self.history

In [ ]:
# 인스턴스 생성 (Calculator 설계도로 calc라는 붕어빵 하나 찍기)
calc = Calculator()
calc.add(1, 2)  # → 3

In [ ]:
# 같은 인스턴스 안에서 history가 누적되는지 확인
calc.history

In [ ]:
calc.add(3, 4)  # → 7

In [ ]:
# 이전 기록까지 다 들어있음 (self.history는 인스턴스 변수라서)
calc.history

## 5. DocumentStore 클래스 만들기 (Mini RAG의 코어)

### 역할
1. **저장(add)**: 문서를 id, content, source로 묶어 저장
2. **카운트(count)**: 저장된 개수 반환
3. **키워드 검색(search)**: 단순 문자열 포함 검색
4. **유사도 검색(retrieve)**: 임베딩 기반 top-k 검색 ← 진짜 RAG

In [ ]:
# Mina 학습 버전 - retrieve 직접 구현
class DocumentStore:
    def __init__(self):
        # 문서 저장용 리스트와 자동 증가 id
        self.documents = []
        self.next_id = 1

    def add(self, content, source='unknown'):
        # 문서를 dict 형태로 저장하고 id 자동 증가
        self.documents.append({
            'id': self.next_id,
            'content': content,
            'source': source
        })
        self.next_id += 1

    def count(self):
        return len(self.documents)

    def search(self, keyword):
        # 단순 부분 문자열 검색 (의미 기반 아님)
        return [doc for doc in self.documents if keyword in doc['content']]

    def retrieve(self, query, count):
        # 질의와 가장 유사도가 높은 문서를 출력
        # ⚠️ 주의: 여기서는 실수로 'source'로 유사도 계산했음 - 실제로는 'content'가 맞음
        contents = [doc['source'] for doc in self.documents]

        query_vec = embeddings.embed_query(query)     # 쿼리 벡터화
        doc_vecs = embeddings.embed_documents(contents)  # 문서 벡터화

        scores = cosine_similarity([query_vec], doc_vecs)[0]

        ranked = sorted(zip(self.documents, scores), key=lambda x: x[1], reverse=True)
        return [(doc, float(score)) for doc, score in ranked][:count]

In [ ]:
# DocumentStore 테스트 - 여러 문서 추가
store = DocumentStore()
store.add('파이썬은 데이터 분석에 좋다', 'guide.txt')
store.add('RAG는 검색 증강 생성이다', 'glossary.txt')
store.add('파이썬은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다. 간결한 문법과 풍부한 라이브러리가 특징입니다.', 'programming_guide.txt')
store.add('RAG는 Retrieval-Augmented Generation의 약자로, LLM이 외부 데이터를 참조하여 답변을 생성하는 기술입니다.', 'ai_glossary.txt')
store.add('벡터 데이터베이스는 텍스트를 숫자 벡터로 변환하여 저장하고, 유사도 검색을 빠르게 수행하는 데이터베이스입니다.', 'database_manual.txt')

In [ ]:
store.count()  # 5개

In [ ]:
# 키워드 검색: '파이썬'이 포함된 문서만
store.search('파이썬')

In [ ]:
# 유사도 검색 - top-1만 반환
query = 'RAG에 대해 설명해 주세요.'
store.retrieve(query, 1)

### 선생님 버전 - 버그 수정 + 예외처리 포함

- `top_k` 기본값을 3으로
- 빈 documents에 대한 예외처리 추가
- 실제로는 `content` 기반 유사도가 맞음 (source 파일명으로 유사도 구하면 의미 없음)

In [ ]:
# 선생님 답안 버전 - 좀 더 안정적인 구현
class DocumentStoreT:
    def __init__(self):
        self.documents = []
        self.next_id = 1

    def add(self, content, source='unknown'):
        self.documents.append({
            'id': self.next_id,
            'content': content,
            'source': source
        })
        self.next_id += 1

    def count(self):
        return len(self.documents)

    def search(self, keyword):
        return [doc for doc in self.documents if keyword in doc['content']]

    def retrieve(self, query, top_k=3):
        # 예외처리: 문서가 없으면 빈 리스트 반환 (에러 방지)
        if not self.documents:
            return []

        contents = [doc['source'] for doc in self.documents]

        query_vec = embeddings.embed_query(query)
        doc_vecs = embeddings.embed_documents(contents)

        scores = cosine_similarity([query_vec], doc_vecs)[0]

        ranked = sorted(zip(self.documents, scores), key=lambda x: x[1], reverse=True)
        return [(doc, float(score)) for doc, score in ranked][:top_k]

## 6. 실습: WordCounter 클래스 (클래스 설계 연습)

- `add_text(text)`: 문자열 저장
- `count_words()`: 지금까지 저장된 모든 문자열의 단어 수 총합
- 단어 기준: `split()` = 공백 기준 쪼개기

In [ ]:
# Mina 버전
class WordCounter:
    def __init__(self):
        self.store = []  # 텍스트 저장소

    def add_text(self, text):
        # 텍스트를 저장하고 단어 수를 알려줌
        self.store.append(text)
        return f'{len(text.split())} 단어 저장완료'

    def count_words(self):
        # 저장된 모든 텍스트의 단어 수 합계
        # split()은 공백 기준으로 단어 분리 (한글이면 어절 단위)
        return sum(len(t.split()) for t in self.store)

# 인스턴스 생성
counter = WordCounter()

In [ ]:
counter.add_text('저 장 해 볼 까 나')  # 공백 기준 6단어

In [ ]:
counter.count_words()

In [ ]:
# 선생님 답안 - Counter 라이브러리로 most_common까지 구현
from collections import Counter

class WordCounterT:
    def __init__(self):
        self.texts = []

    def add_text(self, text):
        self.texts.append(text)

    def count_words(self):
        return sum(len(t.split()) for t in self.texts)

    def most_common(self, n):
        # 모든 텍스트의 단어를 펼쳐서 최빈 단어 n개 반환
        # Counter: 리스트의 원소별 개수를 세주는 딕셔너리 (파이썬 표준 라이브러리)
        all_words = []
        for t in self.texts:
            all_words.extend(t.split())
        return Counter(all_words).most_common(n)

counterT = WordCounterT()

In [ ]:
counterT.add_text('파이썬은 어렵지 않아요~')
counterT.add_text('나는 학교에 갑니다~')
counterT.count_words()
counterT.most_common(3)

## 7. 진짜 RAG: Generation 단계 추가

지금까지는 **Retrieval**만 했다. 이제 검색 결과를 LLM에 넣어 **답변 생성**까지 연결한다.

### 핵심 프롬프트 구조

```
SystemMessage: 제공된 문서를 기반으로 답변하라.
HumanMessage:
  문서:
  - 문서1 내용
  - 문서2 내용
  
  질문: {query}
```

이렇게 **검색된 문서를 컨텍스트로 끼워넣는 것**이 RAG의 본질.

In [ ]:
# LangChain 메시지 객체를 이용한 RAG 함수 (문서 리스트 기반)
def rag_with_langchain(query, documents):
    # 예외처리: 문서가 없으면 안전하게 종료
    if not documents:
        return '참고할 문서가 없습니다.'

    # 검색된 문서들을 하나의 context 문자열로 합치기
    context = '\n'.join(f'- {doc}' for doc in documents)

    # 시스템 메시지 + 휴먼 메시지 구조
    messages = [
        SystemMessage(content='제공된 문서를 기반으로 정확하게 답변해 주세요.'),
        HumanMessage(content=f'문서: \n{context}\n\n질문: {query}\n답변:')
    ]

    # ⚠️ 주의: invoke에는 'messages' (복수형) 리스트를 전달
    response = llm.invoke(messages)
    # LLM이 생성한 최종 문자열 반환
    return response.content

In [ ]:
# RAG 테스트 1: RAG 개념 질문
docs = [
    'RAG는 검색 증강 생성 기술입니다.',
    '외부 문서를 검색해서 LLM 답변에 활용합니다.'
]
rag_with_langchain('rag가 뭐야?', docs)

In [ ]:
# RAG 테스트 2: 도메인이 달라도 context만 넣어주면 잘 답변
docs = [
    '김치찌개에는 두부를 넣고, 계란을 넣어야 맛있어',
    '마지막에 다 끓이고 나면 5분 정도 찬바람에 식혀 먹어야 맛있어.'
]
rag_with_langchain('김치찌개를 맛있게 끓이는 방법은?', docs)

In [ ]:
# 예외처리 테스트
rag_with_langchain('rag가 뭐야?', [])

## 8. DocumentStore2: retrieve + generate 통합

이제 클래스에 `generate()` 메서드를 추가해 **진짜 RAG 클래스**로 완성한다.
출처(source) 정보까지 프롬프트에 포함해 LLM이 근거를 밝힐 수 있게 한다.

In [ ]:
# generate를 포함한 완전체 DocumentStore
class DocumentStore2:
    def __init__(self):
        self.documents = []
        self.next_id = 1

    def add(self, content, source='unknown'):
        self.documents.append({
            'id': self.next_id,
            'content': content,
            'source': source
        })
        self.next_id += 1

    def count(self):
        return len(self.documents)

    def search(self, keyword):
        return [doc for doc in self.documents if keyword in doc['content']]

    def retrieve(self, query, top_k=3):
        # 질의와 가장 유사한 문서 top_k개 반환
        if not self.documents:
            return []

        contents = [doc['source'] for doc in self.documents]

        query_vec = embeddings.embed_query(query)
        doc_vecs = embeddings.embed_documents(contents)

        scores = cosine_similarity([query_vec], doc_vecs)[0]

        ranked = sorted(zip(self.documents, scores), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

    def generate(self, query):
        # 1) 검색 → 2) 컨텍스트 구성 → 3) LLM 호출
        retrieved = self.retrieve(query)
        if not retrieved:
            return '관련 문서가 없습니다.'

        # 출처와 id를 명시해 LLM이 근거를 밝히도록 유도
        context = '\n'.join(
            f"[문서 {doc[0]['id']}, 출처: {doc[0]['source']} {doc[0]['content']}]"
            for doc in retrieved
        )
        messages = [
            SystemMessage(content='제공된 문서를 기반으로 정확하게 답변해 주세요. 출처를 명시해 주세요.'),
            HumanMessage(content=f'문서: \n{context}\n\n질문: {query}\n답변:')
        ]
        response = llm.invoke(messages)
        return response.content

In [ ]:
# RAG 통합 테스트
docs_data = [
    {'id': '1', 'content': 'RAG는 검색 증강 생성 기술입니다.'},
    {'id': '2', 'content': '외부 문서를 검색해서 LLM 답변에 활용합니다.'},
    {'id': '3', 'content': '김치찌개에는 두부를 넣고, 계란을 넣어야 맛있어'},
    {'id': '4', 'content': '마지막에 다 끓이고 나면 5분 정도 찬바람에 식혀 먹어야 맛있어.'}
]

store2 = DocumentStore2()

# 문서를 하나씩 추가
for doc_item in docs_data:
    store2.add(content=doc_item['content'], source=f"doc_{doc_item['id']}")

query = 'RAG에 대해 설명해 주세요?'

# retrieve + generate 한 방에 실행
store2.generate(query)

## 9. Document 객체와 파일 로딩

지금까지는 직접 작성한 딕셔너리를 썼지만, 실무에서는 **파일에서 문서를 로딩**해야 한다.
LangChain은 표준 문서 객체 `Document`를 제공한다.

### Document 구조
- `page_content`: 실제 문서 본문
- `metadata`: 출처, 파일 타입, 글자 수 등 부가 정보 (딕셔너리)

### 비유

> Document = **라벨 붙은 종이**
> - 종이 내용(page_content)
> - 라벨 스티커(metadata): "사내규정.txt, 356자, 2026-03-19 로드됨"

In [ ]:
# Document 객체 생성 예시
doc = Document(
    page_content='문서 내용...',
    metadata={'source': 'file.txt', 'type': 'policy'}
)

In [ ]:
# txt 파일을 직접 읽어보기 (아주 기본)
file_path = SAMPLE_DIR / 'company_policy.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

In [ ]:
file_path.name  # 파일명만 추출

In [ ]:
raw_text  # 파일 내용 확인

In [ ]:
# 파일 내용 + 메타데이터를 Document로 포장
# 이후 vectorstore, retriever에서는 Document 객체를 표준 입력으로 받는다
doc = Document(
    page_content=raw_text,
    metadata={
        'source': file_path.name,
        'type': 'txt',
        'char_count': len(raw_text),       # 글자 수
        'line_count': len(raw_text.splitlines())  # 줄 수
    }
)

In [ ]:
doc

## 10. 디렉토리 내 모든 파일 로딩 - 실전 로더 함수

폴더에서 모든 `.txt` 파일을 읽어 `Document` 리스트로 반환하는 함수를 만든다.
실무에서 `DirectoryLoader`를 쓰기 전에, 원리를 이해하는 단계.

In [ ]:
# 디렉토리의 모든 txt 파일을 Document 리스트로 변환
def load_text_files(directory):
    documents = []
    # Path.glob('*.txt'): 해당 확장자 파일만 필터링 (sorted로 파일명 순 정렬)
    for fp in sorted(directory.glob('*.txt')):
        # ⚠️ encoding='utf=8' 오타 주의 (원본 코드 그대로 - 실제론 utf-8)
        text = fp.read_text(encoding='utf=8')
        doc = Document(
            page_content=text,
            metadata={
                'source': fp.name,
                'char_count': len(text)
            }
        )
        documents.append(doc)
    return documents

In [ ]:
# 전체 파일 로딩 실행
all_docs = load_text_files(SAMPLE_DIR)

In [ ]:
# 3개 파일이 Document로 로딩됨 (ai_report, company_policy, product_manual)
all_docs

## 11. CSV 파일 로딩

CSV는 행 단위로 Document를 만드는 게 일반적이다.
예) 직원 한 명당 하나의 Document → 이름/부서/직급을 `| ` 구분자로 잇기.

> 비유: CSV 한 행 = **명함 한 장**. 각 명함을 Document로 포장해 리스트에 담는다.

In [ ]:
# csv → Document 리스트
def load_csv_file(csv_file_path):
    documents = []
    with open(csv_file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)  # 첫 행을 헤더로 읽음 → 딕셔너리 형태
        for i, row in enumerate(reader):
            # 각 컬럼을 'key: value' 형식으로 이어붙여 하나의 문장으로
            content = ' | '.join(f'{k}: {v}' for k, v in row.items())
            doc = Document(
                page_content=content,
                metadata={
                    'source': csv_file_path.name,
                    'char_count': len(content)
                }
            )
            documents.append(doc)
    return documents

In [ ]:
# 직원 3명 → Document 3개
csv_docs = load_csv_file(SAMPLE_DIR / 'employees.csv')
csv_docs

## 12. JSON 파일 로딩

JSON은 중첩 구조라 방법이 두 가지:
1. **통째로 문자열화**: `json.dumps`로 예쁘게 포맷해서 통으로 저장
2. **키별로 분리**: 각 필드를 별도 Document로 (여기선 1번 방식)

In [ ]:
# 테스트용 json 파일 만들기
test_json = {'name': 'RAG 프로젝트', 'version': '1.0', 'feature': ['검색', '생성']}
with open('sample_data/test.json', 'w', encoding='utf-8') as f:
    # ensure_ascii=False: 한글 그대로 저장 (True면 \uXXXX 유니코드로)
    # indent=2: 들여쓰기로 가독성 증가
    json.dump(test_json, f, ensure_ascii=False, indent=2)

In [ ]:
# json → dict로 읽기, 다시 예쁜 문자열로 변환
with open('sample_data/test.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

content = json.dumps(data, ensure_ascii=False, indent=2)

In [ ]:
data  # 파이썬 dict

In [ ]:
content  # 문자열 (Document의 page_content로 사용 가능)

In [ ]:
# metadata에 keys 정보를 넣어 나중에 검색할 때 힌트로 사용
doc = Document(
    page_content=content,
    metadata={'source': 'test.json', 'keys': list(data.keys())}
)
doc

In [ ]:
# JSON 로더 함수로 정리
def load_json_as_document(file_path):
    path = Path(file_path)
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # JSON 전체를 들여쓰기된 문자열로 → LLM이 구조를 읽기 좋음
    content = json.dumps(data, ensure_ascii=False, indent=2)
    keys = list(data.keys())
    return Document(
        page_content=content,
        metadata={'source': path.name, 'keys': keys}
    )

## 13. 다음 시간 예고 - FAISS 벡터스토어로 확장

오늘 배운 것
- Document 객체로 문서 표준화
- txt/csv/json 로더 작성
- DocumentStore 클래스로 retrieve + generate 통합

다음 시간 (2026-03-20)
- **FAISS 벡터스토어**: 우리가 만든 DocumentStore를 프로덕션급으로 업그레이드
- **LCEL (LangChain Expression Language)**: `prompt | llm | parser` 파이프 문법
- **RAG Chain 완성**: RunnablePassthrough, RunnableParallel로 깔끔한 체인 구성

## Mina 메모

- self가 붙으면 **인스턴스 전체 공유 변수**, 안 붙으면 **지역변수**
- `retrieve`에서 `content`가 아닌 `source`로 유사도 구한 건 실습 중 실수였음 → 실제로는 `content` 기반이 의미적 검색
- `embeddings.embed_query()` vs `embed_documents()`: 전자는 단일, 후자는 배치. OpenAI API 비용과 직결되니 배치를 활용하는 게 경제적
- `ensure_ascii=False`: 한글 깨짐 방지 옵션 - json 쓸 때마다 매번 기억할 것